# Query Optimization Comparison

Setup Cell

In [16]:
import time

import pandas as pd
import pyarrow as pa
import pyarrow.dataset as ds
import pyarrow.fs as pafs

from deltalake import DeltaTable

STORAGE_OPTIONS = {
    "endpoint_url": "http://localhost:9000",
    "aws_access_key_id": "minioadmin",
    "aws_secret_access_key": "minioadmin",
    "aws_region": "us-east-1",
    "allow_http": "true",
    "aws_s3_allow_unsafe_rename": "true",
}

SILVER_URI = "s3://silver/taxi/"

 WITHOUT partition filter (full scan)

In [17]:
print("Reading full silver table (no partition filter)...")

times = []
for i in range(3):
    start = time.perf_counter()
    dt = DeltaTable(SILVER_URI, storage_options=STORAGE_OPTIONS)
    full_table = dt.to_pyarrow_table()
    times.append(time.perf_counter() - start)

time_no_partition  = sum(times) / len(times)
rows_full          = full_table.num_rows

print(f"{'─' * 60}")
print(f"  Without partition filter")
print(f"{'─' * 60}")
print(f"  Rows read  : {rows_full:,}")
print(f"  Run 1      : {times[0]:.3f}s")
print(f"  Run 2      : {times[1]:.3f}s")
print(f"  Run 3      : {times[2]:.3f}s")
print(f"  Average    : {time_no_partition:.3f}s")
print(f"{'─' * 60}")

Reading full silver table (no partition filter)...
────────────────────────────────────────────────────────────
  Without partition filter
────────────────────────────────────────────────────────────
  Rows read  : 35,788,226
  Run 1      : 51.686s
  Run 2      : 89.421s
  Run 3      : 84.581s
  Average    : 75.229s
────────────────────────────────────────────────────────────


Query WITH partition filter (pruned scan)

In [18]:
QUERY_YEAR  = 2023
QUERY_MONTH = 6

print(f"Reading silver table (partition filter: {QUERY_YEAR}-{QUERY_MONTH:02d})...")

times = []
for i in range(3):
    start = time.perf_counter()
    dt = DeltaTable(SILVER_URI, storage_options=STORAGE_OPTIONS)
    filtered_table = dt.to_pyarrow_table(
        filters=[
            ("source_year",  "=", QUERY_YEAR),
            ("source_month", "=", QUERY_MONTH),
        ]
    )
    times.append(time.perf_counter() - start)

time_with_partition  = sum(times) / len(times)
rows_filtered = filtered_table.num_rows

print(f"{'─' * 60}")
print(f"  With partition filter (source_year={QUERY_YEAR}, source_month={QUERY_MONTH})")
print(f"{'─' * 60}")
print(f"  Rows read  : {rows_filtered:,} of {rows_full:,}")
print(f"  Run 1      : {times[0]:.3f}s")
print(f"  Run 2      : {times[1]:.3f}s")
print(f"  Run 3      : {times[2]:.3f}s")
print(f"  Average    : {time_with_partition:.3f}s")
print(f"{'─' * 60}")

Reading silver table (partition filter: 2023-06)...
────────────────────────────────────────────────────────────
  With partition filter (source_year=2023, source_month=6)
────────────────────────────────────────────────────────────
  Rows read  : 3,100,627 of 35,788,226
  Run 1      : 4.870s
  Run 2      : 3.916s
  Run 3      : 3.399s
  Average    : 4.062s
────────────────────────────────────────────────────────────


Comparison table

In [21]:
rows_skipped    = rows_full - rows_filtered
improvement_pct = ((time_no_partition - time_with_partition) / time_no_partition) * 100
speedup         = time_no_partition / time_with_partition

comparison = pd.DataFrame({
    "Metric": [
        "Avg read time",
        "Rows read",
        "Rows skipped",
        "Speedup factor",
        "Improvement",
    ],
    "Without partition filter": [
        f"{time_no_partition:.3f}s",
        f"{rows_full:,} (100%)",
        "0",
        "1.0x",
        "—",
    ],
    "With partition filter": [
        f"{time_with_partition:.3f}s",
        f"{rows_filtered:,} ({rows_filtered/rows_full*100:.0f}%)",
        f"{rows_skipped:,}",
        f"{speedup:.2f}x",
        f"{improvement_pct:.1f}%",
    ],
})

print(f"{'─' * 60}")
print(f"  Partition Pruning Benchmark — Silver Layer")
print(f"  Partition filter: source_year={QUERY_YEAR}, source_month={QUERY_MONTH}")
print(f"{'─' * 60}")
display(comparison)

────────────────────────────────────────────────────────────
  Partition Pruning Benchmark — Silver Layer
  Partition filter: source_year=2023, source_month=6
────────────────────────────────────────────────────────────


,Metric,Without partition filter,With partition filter
0,Avg read time,75.229s,4.062s
1,Rows read,"35,788,226 (100%)","3,100,627 (9%)"
2,Rows skipped,0,"32,687,599"
3,Speedup factor,1.0x,18.52x
4,Improvement,—,94.6%
